# Stride — train the floor-plan recognition model (Colab)

Open this in Colab, set the runtime to a **GPU** (Runtime → Change runtime type → T4 GPU), then **Runtime → Run all**.

It clones the repo, generates the synthetic dataset, trains the U‑Net, exports ONNX, and downloads `best.pt` + `stride-planseg.onnx` to your computer.

The free T4 handles this comfortably. A full run (10k samples, 30 epochs) is roughly 2–4 hours. To do a quick end‑to‑end test first, lower `SAMPLES` and `EPOCHS` in the config cell.

## 1. Check the GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('\n⚠️  No GPU. Runtime → Change runtime type → Hardware accelerator: T4 GPU, then Run all again.')

## 2. Config — tweak these, then Run all

In [ ]:
BRANCH  = 'main-uiyymm'   # branch to train from
SAMPLES = 10000           # training plans to generate (try 400 for a quick test)
VAL     = 500             # validation plans
EPOCHS  = 30              # training epochs (try 3 for a quick test)
BATCH   = 8               # lower to 4 if you hit out-of-memory
SIZE    = 512             # training crop size
BASE    = 32              # U-Net width (model capacity)

## 3. Clone the repo + install Node deps (for the generator)

In [ ]:
import os
if not os.path.isdir('stride'):
    !git clone --branch {BRANCH} https://github.com/tiienn/stride.git
%cd stride
!node --version
# only the generator's dep is needed (resvg); skip the app's heavy 3D deps
!npm install @resvg/resvg-js --no-save --no-audit --no-fund

## 4. Generate the synthetic dataset

Images + pixel‑perfect masks + ground‑truth JSON. ~110 ms/sample.

In [ ]:
!node ml/generate.mjs --count {SAMPLES} --out ml/data/train --seed 1
!node ml/generate.mjs --count {VAL} --out ml/data/val --seed 999
import glob
print('train images:', len(glob.glob('ml/data/train/img_*.png')))
print('val images:  ', len(glob.glob('ml/data/val/img_*.png')))

## 5. Preview a sample (sanity check)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open('ml/data/train/img_00000.png')); ax[0].set_title('drawing'); ax[0].axis('off')
ax[1].imshow(Image.open('ml/data/train/msk_00000.png')); ax[1].set_title('mask: red=wall green=door blue=window'); ax[1].axis('off')
plt.show()

## 6. Train

Watch the per‑class `val IoU`. Walls climb first; doors/windows lag (rarer pixels) but the class weights compensate. Good v1 targets: wall ≥ 0.85, door/window ≥ 0.6. `best.pt` is saved whenever val mIoU improves.

In [ ]:
%cd /content/stride/ml/train
!python train.py --data ../data/train --out ../checkpoints \
    --epochs {EPOCHS} --batch {BATCH} --size {SIZE} --base {BASE} --val-frac 0.05
%cd /content/stride

## 7. Try it on a held‑out plan

In [ ]:
%cd /content/stride/ml/train
!python infer.py --checkpoint ../checkpoints/best.pt --base {BASE} \
    --image ../data/val/img_00003.png --out-prefix /content/pred
%cd /content/stride
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
ax[0].imshow(Image.open('ml/data/val/img_00003.png')); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(Image.open('/content/pred_mask.png')); ax[1].set_title('model prediction'); ax[1].axis('off')
plt.show()

## 8. Export ONNX (for in‑browser inference in Stride)

In [ ]:
!pip install --quiet onnx
%cd /content/stride/ml/train
import torch
from model import UNet
m = UNet(4, base=BASE)
m.load_state_dict(torch.load('../checkpoints/best.pt', map_location='cpu'))
m.eval()
torch.onnx.export(m, torch.zeros(1, 3, 512, 512), '/content/stride-planseg.onnx',
                  input_names=['image'], output_names=['logits'], opset_version=17,
                  dynamo=False, dynamic_axes={'image': {2: 'h', 3: 'w'}, 'logits': {2: 'h', 3: 'w'}})
%cd /content/stride
import os
print('ONNX size: %.1f MB' % (os.path.getsize('/content/stride-planseg.onnx') / 1e6))

## 9. Download the trained model

In [ ]:
from google.colab import files
files.download('/content/stride/ml/checkpoints/best.pt')
files.download('/content/stride-planseg.onnx')